Collaborative Filtering and Memory Modeling

In [1]:
import pandas as pd
import numpy as np

In [2]:
header = ['user_id','item_id','rating','timestamp']
df = pd.read_csv('u.data', sep='\t', names=header)
df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [3]:
n_users = df.user_id.unique().shape[0]
n_items = df.item_id.unique().shape[0]
print('Num of user =' + str(n_users) + '| Num of item = ' + str(n_items))

Num of user =943| Num of item = 1682


In [4]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, test_size= 0.25)

In [36]:
train_data_mat = np.zeros((n_users, n_items))
for line in train_data.itertuples():
     train_data_mat[line [1]-1, line[2]-1] = line[3]

In [35]:
test_data_mat = np.zeros((n_users, n_items))
for line in test_data.itertuples():
     test_data_mat[line [1]-1, line[2]-1] = line[3]

In [29]:
def predict(ratings, similarity, kind="user"):
    if kind == 'user':
        mean_user_rating = ratings.mean(axis=1)
        ratings_diff = ratings - mean_user_rating[:, np.newaxis]
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
    elif kind == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])
    return pred

In [30]:
from sklearn.metrics.pairwise import pairwise_distances
user_sim = pairwise_distances(train_data_mat)
item_sim = pairwise_distances(train_data_mat.T)

In [31]:
item_prediction = predict(train_data_mat, item_sim, kind='item')
user_prediction = predict(train_data_mat, user_sim, kind='user')

In [41]:
from sklearn.metrics import mean_squared_error
from math import sqrt

def rsme(prediction, ground_truth):
    prediction = prediction[ground_truth.nonzero()].flatten()
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()
    return sqrt(mean_squared_error(prediction, ground_truth))

print('User-based cf rsme :' + str(rsme(user_prediction, test_data_mat)))
print('Item-based cf rsme :' + str(rsme(item_prediction, test_data_mat)))

User-based cf rsme :3.068183734624382
Item-based cf rsme :3.3436582562738963
